# Used Cars: exploratory data analysis

This notebook answers descriptive business questions about Craigslist used-car listings. It deliberately does **not** include precomputed outputs: provide the source data and run it locally to generate current results.

The notebook never imputes values with statistics calculated from a hidden evaluation set and contains no predictive model, so train/test leakage is not applicable to this EDA. If modeling is added later, imputation and feature engineering must be fitted on the training split only.


In [ ]:
import os
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

sns.set_theme(style="whitegrid")


def resolve_data_path() -> Path:
    """Resolve the input CSV without assuming a machine-specific absolute path."""
    configured = os.environ.get("USED_CARS_DATA", "").strip()
    candidates = [Path(configured).expanduser()] if configured else [
        Path.cwd() / "data/vehicles.csv",
        Path.cwd() / "vehicles.csv",
    ]
    for candidate in candidates:
        if candidate.is_file():
            return candidate.resolve()
    expected = " or ".join(str(path) for path in candidates)
    raise FileNotFoundError(
        "Used-car data was not found. Place vehicles.csv under data/ or set "
        f"USED_CARS_DATA to a CSV file path (checked: {expected})."
    )


DATA_PATH = resolve_data_path()
print(f"Loading data from {DATA_PATH}")


In [ ]:
raw_df = pd.read_csv(DATA_PATH)
required_columns = {
    "id",
    "price",
    "year",
    "condition",
    "title_status",
    "type",
    "odometer",
    "fuel",
    "posting_date",
}
missing_columns = sorted(required_columns.difference(raw_df.columns))
if missing_columns:
    raise ValueError(
        "The input CSV is missing required columns: " + ", ".join(missing_columns)
    )
if raw_df.empty:
    raise ValueError("The input CSV contains no rows to analyze.")

# Keep raw values intact where possible; coerce only fields used by the analysis.
for column in ("price", "year", "odometer", "lat", "long"):
    if column in raw_df.columns:
        raw_df[column] = pd.to_numeric(raw_df[column], errors="coerce")
        raw_df[column] = raw_df[column].replace([np.inf, -np.inf], np.nan)
raw_df["posting_date"] = pd.to_datetime(
    raw_df["posting_date"], errors="coerce", format="mixed", utc=True
)

# Exact non-missing listing IDs are the safest available de-duplication key.
# Rows without an ID are retained because they cannot be proven to be duplicates.
rows_before = len(raw_df)
with_ids = raw_df[raw_df["id"].notna()].drop_duplicates(subset=["id"], keep="first")
without_ids = raw_df[raw_df["id"].isna()]
analysis_df = pd.concat([with_ids, without_ids]).sort_index().reset_index(drop=True)
for column in ("condition", "title_status", "type", "fuel"):
    analysis_df[column] = analysis_df[column].astype("string").str.strip().replace("", pd.NA)
print(f"Rows loaded: {rows_before:,}; rows retained for analysis: {len(analysis_df):,}")


In [ ]:
quality_report = pd.DataFrame(
    {
        "dtype": analysis_df.dtypes.astype(str),
        "missing": analysis_df.isna().sum(),
        "missing_pct": (analysis_df.isna().mean() * 100).round(2),
        "unique": analysis_df.nunique(dropna=True),
    }
).sort_values(["missing_pct", "missing"], ascending=False)
print(quality_report.to_string())


## Business question 1: What vehicle types are most common?


In [ ]:
type_counts = analysis_df["type"].dropna().astype("string").value_counts()
if type_counts.empty:
    raise ValueError("The type column has no usable values after loading the data.")
print(type_counts.head(10))

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
type_counts.head(10).plot(kind="bar", ax=axes[0], color="steelblue")
axes[0].set(title="Top vehicle types", xlabel="Vehicle type", ylabel="Listings")
type_counts.head(8).plot(kind="pie", autopct="%1.1f%%", ax=axes[1], ylabel="")
axes[1].set_title("Share among the eight most common types")
plt.tight_layout()
plt.show()


## Business question 2: How are listings distributed by title status?


In [ ]:
title_counts = analysis_df["title_status"].dropna().astype("string").value_counts()
print(title_counts)

ax = title_counts.plot(kind="bar", figsize=(10, 5), color="darkorange")
ax.set(title="Listings by title status", xlabel="Title status", ylabel="Listings")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()


## Business question 3: What is the most common car condition?


In [ ]:
condition_counts = analysis_df["condition"].dropna().astype("string").value_counts()
if condition_counts.empty:
    print("No non-missing condition values are available.")
else:
    print(f"Most common condition: {condition_counts.index[0]}")
    condition_counts.plot(kind="bar", figsize=(9, 5), color="firebrick")
    plt.title("Listings by vehicle condition")
    plt.xlabel("Condition")
    plt.ylabel("Listings")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.show()


## Business question 4: What is the distribution of listings over vehicle year?

This is the vehicle manufacture year, not the listing date. Invalid or missing years are excluded from this chart rather than replaced with a mean.


In [ ]:
year_values = analysis_df["year"].dropna()
current_year = datetime.now(timezone.utc).year
valid_years = year_values[
    year_values.ge(1886)
    & year_values.le(current_year)
    & year_values.mod(1).eq(0)
]
year_counts = valid_years.astype("int64").value_counts().sort_index()
if year_counts.empty:
    print("No usable manufacture-year values are available.")
else:
    year_counts.plot.area(figsize=(14, 6), color="seagreen", alpha=0.45)
    plt.title("Listings by vehicle manufacture year")
    plt.xlabel("Manufacture year")
    plt.ylabel("Listings")
    plt.tight_layout()
    plt.show()


## Business question 5: What is the relationship between mileage and price?

A deterministic sample keeps the scatter plot readable. It is not used for estimation or model evaluation. Negative values are excluded because they are invalid measurements.


In [ ]:
mileage_price = analysis_df[["odometer", "price"]].dropna()
mileage_price = mileage_price[(mileage_price["odometer"] >= 0) & (mileage_price["price"] >= 0)]
if mileage_price.empty:
    print("No valid non-negative mileage/price pairs are available.")
else:
    plot_data = mileage_price.sample(min(len(mileage_price), 10_000), random_state=42)
    plt.figure(figsize=(11, 6))
    sns.scatterplot(data=plot_data, x="odometer", y="price", alpha=0.25, s=18, edgecolor=None)
    plt.title("Listed price versus odometer reading")
    plt.xlabel("Odometer")
    plt.ylabel("Listed price")
    plt.tight_layout()
    plt.show()


## Business question 6: How do prices compare by fuel type?

Means are shown together with listing counts; a mean alone can be distorted by outliers and should not be interpreted as a causal price effect.


In [ ]:
fuel_price = analysis_df[["fuel", "price"]].dropna()
fuel_price = fuel_price[fuel_price["price"] >= 0]
if fuel_price.empty:
    print("No valid fuel/price pairs are available.")
else:
    fuel_summary = (
        fuel_price.groupby("fuel", observed=True)["price"]
        .agg(listings="size", mean_price="mean", median_price="median")
        .sort_values("mean_price")
    )
    print(fuel_summary.round(2))
    fuel_summary["mean_price"].plot(kind="bar", figsize=(10, 5), color="goldenrod")
    plt.title("Average listed price by fuel type")
    plt.xlabel("Fuel type")
    plt.ylabel("Average listed price")
    plt.xticks(rotation=30, ha="right")
    plt.tight_layout()
    plt.show()


## Interpretation guardrails

- These are descriptive listing-level comparisons, not causal estimates.
- The repository does not include the source data, so no numerical findings are claimed here.
- For a future predictive model, use a reproducible train/test split and fit all preprocessing steps on the training partition only.
